In [4]:
# =============================================================================
# MOSDAC INSAT HEM DAILY PRECIPITATION ANALYSIS
# =============================================================================
#
# Product:
#   3RIMG_*_L3B_HEM_DLY_V01R00.h5
#
# Period:
#   10 July 2026 - 19 July 2026
#
# Rainfall:
#   HEM_DLY
#   units = mm/day
#
# Latitude:
#   Latitude
#   scale_factor = 0.01
#
# Longitude:
#   Longitude
#   scale_factor = 0.01
#
# =============================================================================

import os
import glob
import warnings

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


# =============================================================================
# USER SETTINGS
# =============================================================================

BASE_DIR = r"Z:\1\MON FLOOD\MOSDAC_Data\MOSDAC\2026"

START_DATE = "2026-07-10"
END_DATE   = "2026-07-19"


# =============================================================================
# OUTPUT DIRECTORIES
# =============================================================================

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "MOSDAC_HEM_RAINFALL_ANALYSIS_10JUL_19JUL_2026"
)

DAILY_MAP_DIR = os.path.join(
    OUTPUT_DIR,
    "01_DAILY_SPATIAL_MAPS"
)

SUMMARY_DIR = os.path.join(
    OUTPUT_DIR,
    "02_SUMMARY"
)

CSV_DIR = os.path.join(
    OUTPUT_DIR,
    "03_CSV"
)

NPY_DIR = os.path.join(
    OUTPUT_DIR,
    "04_NUMPY"
)

for folder in [
    OUTPUT_DIR,
    DAILY_MAP_DIR,
    SUMMARY_DIR,
    CSV_DIR,
    NPY_DIR
]:

    os.makedirs(
        folder,
        exist_ok=True
    )


# =============================================================================
# MOSDAC DATASET NAMES
# =============================================================================

RAIN_DATASET = "HEM_DLY"

LAT_DATASET = "Latitude"

LON_DATASET = "Longitude"


# =============================================================================
# MOSDAC DEFAULT VALUES
# =============================================================================

RAIN_FILL_VALUE = -999.0

LAT_FILL_VALUE = 32767

LON_FILL_VALUE = 32767


# =============================================================================
# MOSDAC SCALE FACTORS
# =============================================================================

LAT_SCALE = 0.01

LON_SCALE = 0.01


# =============================================================================
# MAP SETTINGS
# =============================================================================

DPI = 250

# Minimum rainfall value
FIXED_VMIN = 0

# None = automatically determine common scale
FIXED_VMAX = None


# =============================================================================
# FIND DAILY FILES
# =============================================================================

def find_daily_files():

    start = pd.Timestamp(
        START_DATE
    )

    end = pd.Timestamp(
        END_DATE
    )

    files = []

    current = start

    while current <= end:

        folder_name = current.strftime(
            "%d%b"
        ).upper()

        folder = os.path.join(
            BASE_DIR,
            folder_name
        )

        print(
            f"\nChecking folder:"
        )

        print(folder)

        if not os.path.isdir(folder):

            print(
                "WARNING: Folder does not exist."
            )

        else:

            found = glob.glob(
                os.path.join(
                    folder,
                    "*.h5"
                )
            )

            # Remove duplicate paths
            found = sorted(
                set(
                    os.path.abspath(f)
                    for f in found
                )
            )

            hem_files = [
                f
                for f in found
                if "_HEM_DLY_" in os.path.basename(f)
            ]

            if hem_files:

                files.extend(
                    hem_files
                )

                print(
                    f"Found {len(hem_files)} HEM file(s)."
                )

            else:

                print(
                    "WARNING: No HEM daily file found."
                )

        current += pd.Timedelta(
            days=1
        )

    # Final duplicate removal
    files = sorted(
        set(files)
    )

    return files


# =============================================================================
# EXTRACT DATE FROM FILENAME
# =============================================================================

def extract_date_from_filename(
    filename
):

    basename = os.path.basename(
        filename
    )

    try:

        date_part = basename.split("_")[1]

        return pd.to_datetime(
            date_part,
            format="%d%b%Y"
        )

    except Exception:

        return None


# =============================================================================
# READ MOSDAC HDF5
# =============================================================================

def read_mosdac_file(
    filename
):

    print("\n")
    print("=" * 90)
    print("READING FILE")
    print("=" * 90)

    print(filename)

    with h5py.File(
        filename,
        "r"
    ) as h5:

        # ---------------------------------------------------------------------
        # Check datasets
        # ---------------------------------------------------------------------

        for dataset_name in [
            RAIN_DATASET,
            LAT_DATASET,
            LON_DATASET
        ]:

            if dataset_name not in h5:

                raise KeyError(
                    f"Dataset '{dataset_name}' "
                    f"not found."
                )

        # ---------------------------------------------------------------------
        # RAINFALL
        # ---------------------------------------------------------------------

        rain_ds = h5[
            RAIN_DATASET
        ]

        rainfall = rain_ds[:]

        rainfall = np.squeeze(
            rainfall
        )

        rainfall = rainfall.astype(
            np.float64
        )

        rain_fill = rain_ds.attrs.get(
            "_FillValue",
            RAIN_FILL_VALUE
        )

        rain_fill = float(
            np.asarray(
                rain_fill
            ).flatten()[0]
        )

        rainfall[
            rainfall == rain_fill
        ] = np.nan

        rainfall[
            ~np.isfinite(rainfall)
        ] = np.nan

        # Negative precipitation is invalid
        rainfall[
            rainfall < 0
        ] = np.nan

        # ---------------------------------------------------------------------
        # LATITUDE
        # ---------------------------------------------------------------------

        lat_ds = h5[
            LAT_DATASET
        ]

        latitude = lat_ds[:].astype(
            np.float64
        )

        lat_fill = lat_ds.attrs.get(
            "_FillValue",
            LAT_FILL_VALUE
        )

        lat_fill = float(
            np.asarray(
                lat_fill
            ).flatten()[0]
        )

        lat_scale = lat_ds.attrs.get(
            "scale_factor",
            LAT_SCALE
        )

        lat_scale = float(
            np.asarray(
                lat_scale
            ).flatten()[0]
        )

        lat_offset = lat_ds.attrs.get(
            "add_offset",
            0.0
        )

        lat_offset = float(
            np.asarray(
                lat_offset
            ).flatten()[0]
        )

        latitude[
            latitude == lat_fill
        ] = np.nan

        latitude = (
            latitude * lat_scale
            +
            lat_offset
        )

        latitude[
            ~np.isfinite(latitude)
        ] = np.nan

        # ---------------------------------------------------------------------
        # LONGITUDE
        # ---------------------------------------------------------------------

        lon_ds = h5[
            LON_DATASET
        ]

        longitude = lon_ds[:].astype(
            np.float64
        )

        lon_fill = lon_ds.attrs.get(
            "_FillValue",
            LON_FILL_VALUE
        )

        lon_fill = float(
            np.asarray(
                lon_fill
            ).flatten()[0]
        )

        lon_scale = lon_ds.attrs.get(
            "scale_factor",
            LON_SCALE
        )

        lon_scale = float(
            np.asarray(
                lon_scale
            ).flatten()[0]
        )

        lon_offset = lon_ds.attrs.get(
            "add_offset",
            0.0
        )

        lon_offset = float(
            np.asarray(
                lon_offset
            ).flatten()[0]
        )

        longitude[
            longitude == lon_fill
        ] = np.nan

        longitude = (
            longitude * lon_scale
            +
            lon_offset
        )

        longitude[
            ~np.isfinite(longitude)
        ] = np.nan

        # ---------------------------------------------------------------------
        # DIMENSION CHECK
        # ---------------------------------------------------------------------

        if rainfall.shape != latitude.shape:

            raise ValueError(
                f"Rainfall shape {rainfall.shape} "
                f"does not match latitude "
                f"{latitude.shape}"
            )

        if rainfall.shape != longitude.shape:

            raise ValueError(
                f"Rainfall shape {rainfall.shape} "
                f"does not match longitude "
                f"{longitude.shape}"
            )

        # ---------------------------------------------------------------------
        # INFORMATION
        # ---------------------------------------------------------------------

        print(
            f"Rainfall shape: {rainfall.shape}"
        )

        print(
            f"Latitude shape: {latitude.shape}"
        )

        print(
            f"Longitude shape: {longitude.shape}"
        )

        print(
            f"Rainfall units: "
            f"{rain_ds.attrs.get('units', 'unknown')}"
        )

        return (
            rainfall,
            latitude,
            longitude
        )


# =============================================================================
# FIND VALID GEOGRAPHIC WINDOW
# =============================================================================

def get_valid_geo_window(
    latitude,
    longitude
):

    # Both coordinate arrays must be finite
    geo_valid = (
        np.isfinite(latitude)
        &
        np.isfinite(longitude)
    )

    # -------------------------------------------------------------------------
    # Find rows containing valid coordinates
    # -------------------------------------------------------------------------

    valid_rows = np.where(
        np.any(
            geo_valid,
            axis=1
        )
    )[0]

    # -------------------------------------------------------------------------
    # Find columns containing valid coordinates
    # -------------------------------------------------------------------------

    valid_cols = np.where(
        np.any(
            geo_valid,
            axis=0
        )
    )[0]

    if (
        valid_rows.size == 0
        or
        valid_cols.size == 0
    ):

        raise ValueError(
            "No valid latitude/longitude "
            "coordinates found."
        )

    row_min = valid_rows.min()

    row_max = valid_rows.max()

    col_min = valid_cols.min()

    col_max = valid_cols.max()

    return (
        row_min,
        row_max,
        col_min,
        col_max
    )


# =============================================================================
# CROP TO VALID GEOGRAPHIC EXTENT
# =============================================================================

def crop_to_valid_geometry(
    rainfall,
    latitude,
    longitude
):

    (
        row_min,
        row_max,
        col_min,
        col_max
    ) = get_valid_geo_window(
        latitude,
        longitude
    )

    rainfall_crop = rainfall[
        row_min:row_max + 1,
        col_min:col_max + 1
    ]

    latitude_crop = latitude[
        row_min:row_max + 1,
        col_min:col_max + 1
    ]

    longitude_crop = longitude[
        row_min:row_max + 1,
        col_min:col_max + 1
    ]

    # -------------------------------------------------------------------------
    # Remove any remaining rows/columns that still contain non-finite
    # coordinate values.
    # -------------------------------------------------------------------------

    coordinate_valid = (
        np.isfinite(latitude_crop)
        &
        np.isfinite(longitude_crop)
    )

    valid_rows = np.where(
        np.all(
            coordinate_valid,
            axis=1
        )
    )[0]

    valid_cols = np.where(
        np.all(
            coordinate_valid,
            axis=0
        )
    )[0]

    # If complete valid rows/columns exist, use them.
    if (
        valid_rows.size > 0
        and
        valid_cols.size > 0
    ):

        rainfall_crop = rainfall_crop[
            valid_rows.min():
            valid_rows.max() + 1,
            valid_cols.min():
            valid_cols.max() + 1
        ]

        latitude_crop = latitude_crop[
            valid_rows.min():
            valid_rows.max() + 1,
            valid_cols.min():
            valid_cols.max() + 1
        ]

        longitude_crop = longitude_crop[
            valid_rows.min():
            valid_rows.max() + 1,
            valid_cols.min():
            valid_cols.max() + 1
        ]

    # -------------------------------------------------------------------------
    # Final coordinate validation
    # -------------------------------------------------------------------------

    final_valid = (
        np.isfinite(latitude_crop)
        &
        np.isfinite(longitude_crop)
    )

    if not np.all(final_valid):

        # This should rarely be necessary.
        # Replace problematic rainfall cells with NaN.
        rainfall_crop[
            ~final_valid
        ] = np.nan

        # For pcolormesh coordinates themselves, use nearest valid
        # values along each problematic row/column.

        latitude_crop = fill_coordinate_array(
            latitude_crop
        )

        longitude_crop = fill_coordinate_array(
            longitude_crop
        )

    return (
        rainfall_crop,
        latitude_crop,
        longitude_crop
    )


# =============================================================================
# FILL REMAINING COORDINATE NaNs
# =============================================================================

def fill_coordinate_array(
    arr
):

    result = arr.copy()

    # -------------------------------------------------------------------------
    # First fill each row using interpolation
    # -------------------------------------------------------------------------

    for i in range(
        result.shape[0]
    ):

        row = result[i, :]

        good = np.isfinite(
            row
        )

        if good.sum() >= 2:

            x = np.arange(
                row.size
            )

            row[~good] = np.interp(
                x[~good],
                x[good],
                row[good]
            )

            result[i, :] = row

    # -------------------------------------------------------------------------
    # Then fill columns if needed
    # -------------------------------------------------------------------------

    for j in range(
        result.shape[1]
    ):

        col = result[:, j]

        good = np.isfinite(
            col
        )

        if good.sum() >= 2:

            y = np.arange(
                col.size
            )

            col[~good] = np.interp(
                y[~good],
                y[good],
                col[good]
            )

            result[:, j] = col

    return result


# =============================================================================
# CALCULATE RAINFALL STATISTICS
# =============================================================================

def calculate_statistics(
    rainfall,
    latitude,
    longitude
):

    valid_mask = (
        np.isfinite(rainfall)
        &
        np.isfinite(latitude)
        &
        np.isfinite(longitude)
    )

    valid = rainfall[
        valid_mask
    ]

    if valid.size == 0:

        return {

            "mean_rainfall_mm_day": np.nan,

            "median_rainfall_mm_day": np.nan,

            "minimum_rainfall_mm_day": np.nan,

            "maximum_rainfall_mm_day": np.nan,

            "std_rainfall_mm_day": np.nan,

            "valid_pixels": 0,

            "total_pixels": rainfall.size,

            "coverage_percent": 0
        }

    return {

        "mean_rainfall_mm_day":
            float(
                np.mean(valid)
            ),

        "median_rainfall_mm_day":
            float(
                np.median(valid)
            ),

        "minimum_rainfall_mm_day":
            float(
                np.min(valid)
            ),

        "maximum_rainfall_mm_day":
            float(
                np.max(valid)
            ),

        "std_rainfall_mm_day":
            float(
                np.std(valid)
            ),

        "valid_pixels":
            int(
                valid.size
            ),

        "total_pixels":
            int(
                rainfall.size
            ),

        "coverage_percent":
            float(
                valid.size /
                rainfall.size *
                100
            )
    }


# =============================================================================
# DETERMINE COMMON MAP SCALE
# =============================================================================

def determine_common_vmax(
    files
):

    if FIXED_VMAX is not None:

        return FIXED_VMAX

    print("\n")
    print("=" * 90)
    print("DETERMINING COMMON MAP SCALE")
    print("=" * 90)

    p99_values = []

    for filename in files:

        try:

            (
                rainfall,
                latitude,
                longitude
            ) = read_mosdac_file(
                filename
            )

            valid = rainfall[
                np.isfinite(rainfall)
            ]

            if valid.size > 0:

                p99 = np.percentile(
                    valid,
                    99
                )

                p99_values.append(
                    p99
                )

                print(
                    f"{os.path.basename(filename)}"
                )

                print(
                    f"  P99 rainfall = "
                    f"{p99:.3f} mm/day"
                )

        except Exception as e:

            print(
                f"WARNING: {e}"
            )

    if not p99_values:

        return 100.0

    vmax = max(
        p99_values
    )

    # -------------------------------------------------------------------------
    # Round to sensible value
    # -------------------------------------------------------------------------

    if vmax <= 10:

        vmax = np.ceil(
            vmax
        )

    elif vmax <= 50:

        vmax = np.ceil(
            vmax / 5
        ) * 5

    elif vmax <= 100:

        vmax = np.ceil(
            vmax / 10
        ) * 10

    elif vmax <= 500:

        vmax = np.ceil(
            vmax / 25
        ) * 25

    else:

        vmax = np.ceil(
            vmax / 50
        ) * 50

    return float(
        vmax
    )


# =============================================================================
# PLOT DAILY MAP
# =============================================================================

def plot_daily_map(
    rainfall,
    latitude,
    longitude,
    date,
    output_file,
    vmin,
    vmax
):

    # -------------------------------------------------------------------------
    # Crop to valid geographic region
    # -------------------------------------------------------------------------

    (
        rainfall_plot,
        latitude_plot,
        longitude_plot
    ) = crop_to_valid_geometry(
        rainfall,
        latitude,
        longitude
    )

    # -------------------------------------------------------------------------
    # Make invalid rainfall pixels NaN
    # -------------------------------------------------------------------------

    rainfall_plot[
        ~np.isfinite(rainfall_plot)
    ] = np.nan

    # -------------------------------------------------------------------------
    # Plot
    # -------------------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(12, 9)
    )

    mesh = ax.pcolormesh(
        longitude_plot,
        latitude_plot,
        rainfall_plot,
        shading="auto",
        vmin=vmin,
        vmax=vmax
    )

    # -------------------------------------------------------------------------
    # Colorbar
    # -------------------------------------------------------------------------

    cbar = plt.colorbar(
        mesh,
        ax=ax,
        pad=0.02,
        shrink=0.85
    )

    cbar.set_label(
        "Daily precipitation (mm/day)",
        fontsize=11
    )

    # -------------------------------------------------------------------------
    # Labels
    # -------------------------------------------------------------------------

    ax.set_xlabel(
        "Longitude (°E)",
        fontsize=11
    )

    ax.set_ylabel(
        "Latitude (°N)",
        fontsize=11
    )

    ax.set_title(
        "MOSDAC INSAT HEM Daily Precipitation\n"
        f"{date.strftime('%d %B %Y')}",
        fontsize=14,
        fontweight="bold"
    )

    # -------------------------------------------------------------------------
    # Geographic extent
    # -------------------------------------------------------------------------

    valid_geo = (
        np.isfinite(latitude_plot)
        &
        np.isfinite(longitude_plot)
    )

    if np.any(valid_geo):

        xmin = np.nanmin(
            longitude_plot[
                valid_geo
            ]
        )

        xmax = np.nanmax(
            longitude_plot[
                valid_geo
            ]
        )

        ymin = np.nanmin(
            latitude_plot[
                valid_geo
            ]
        )

        ymax = np.nanmax(
            latitude_plot[
                valid_geo
            ]
        )

        ax.set_xlim(
            xmin,
            xmax
        )

        ax.set_ylim(
            ymin,
            ymax
        )

    # -------------------------------------------------------------------------
    # Grid
    # -------------------------------------------------------------------------

    ax.grid(
        True,
        alpha=0.3,
        linestyle="--",
        linewidth=0.5
    )

    # -------------------------------------------------------------------------
    # Statistics
    # -------------------------------------------------------------------------

    valid_rain = rainfall[
        np.isfinite(rainfall)
    ]

    if valid_rain.size > 0:

        mean_val = np.mean(
            valid_rain
        )

        median_val = np.median(
            valid_rain
        )

        max_val = np.max(
            valid_rain
        )

        text = (
            f"Mean: {mean_val:.2f} mm/day\n"
            f"Median: {median_val:.2f} mm/day\n"
            f"Maximum: {max_val:.2f} mm/day"
        )

        ax.text(
            0.02,
            0.02,
            text,
            transform=ax.transAxes,
            fontsize=10,
            verticalalignment="bottom",
            bbox=dict(
                boxstyle="round",
                facecolor="white",
                alpha=0.85
            )
        )

    # -------------------------------------------------------------------------
    # Save
    # -------------------------------------------------------------------------

    plt.tight_layout()

    plt.savefig(
        output_file,
        dpi=DPI,
        bbox_inches="tight"
    )

    plt.close()

    print(
        f"Map saved:"
    )

    print(
        output_file
    )


# =============================================================================
# DAILY MEAN TIME SERIES
# =============================================================================

def plot_mean_timeseries(
    df,
    output_file
):

    fig, ax = plt.subplots(
        figsize=(12, 6)
    )

    ax.plot(
        df["date"],
        df["mean_rainfall_mm_day"],
        marker="o",
        linewidth=2,
        markersize=7
    )

    ax.set_xlabel(
        "Date"
    )

    ax.set_ylabel(
        "Spatial mean precipitation (mm/day)"
    )

    ax.set_title(
        "MOSDAC Daily Spatial Mean Precipitation\n"
        "10–19 July 2026",
        fontsize=14,
        fontweight="bold"
    )

    ax.grid(
        True,
        alpha=0.3,
        linestyle="--"
    )

    plt.xticks(
        rotation=45
    )

    plt.tight_layout()

    plt.savefig(
        output_file,
        dpi=DPI,
        bbox_inches="tight"
    )

    plt.close()


# =============================================================================
# DAILY MAXIMUM TIME SERIES
# =============================================================================

def plot_max_timeseries(
    df,
    output_file
):

    fig, ax = plt.subplots(
        figsize=(12, 6)
    )

    ax.plot(
        df["date"],
        df["maximum_rainfall_mm_day"],
        marker="o",
        linewidth=2,
        markersize=7
    )

    ax.set_xlabel(
        "Date"
    )

    ax.set_ylabel(
        "Maximum precipitation (mm/day)"
    )

    ax.set_title(
        "MOSDAC Daily Maximum Precipitation\n"
        "10–19 July 2026",
        fontsize=14,
        fontweight="bold"
    )

    ax.grid(
        True,
        alpha=0.3,
        linestyle="--"
    )

    plt.xticks(
        rotation=45
    )

    plt.tight_layout()

    plt.savefig(
        output_file,
        dpi=DPI,
        bbox_inches="tight"
    )

    plt.close()


# =============================================================================
# CREATE MULTI-DAY SPATIAL PRODUCTS
# =============================================================================

def create_multiday_maps(
    rainfall_stack,
    latitude,
    longitude,
    vmax
):

    print("\n")
    print("=" * 90)
    print("CREATING MULTI-DAY SPATIAL PRODUCTS")
    print("=" * 90)

    stack = np.stack(
        rainfall_stack,
        axis=0
    )

    # -------------------------------------------------------------------------
    # Mean daily rainfall
    # -------------------------------------------------------------------------

    period_mean = np.nanmean(
        stack,
        axis=0
    )

    # -------------------------------------------------------------------------
    # Maximum daily rainfall
    # -------------------------------------------------------------------------

    period_max = np.nanmax(
        stack,
        axis=0
    )

    # -------------------------------------------------------------------------
    # Accumulated rainfall
    # -------------------------------------------------------------------------

    period_total = np.nansum(
        stack,
        axis=0
    )

    # -------------------------------------------------------------------------
    # Save arrays
    # -------------------------------------------------------------------------

    np.save(
        os.path.join(
            NPY_DIR,
            "Mean_Daily_Rainfall.npy"
        ),
        period_mean
    )

    np.save(
        os.path.join(
            NPY_DIR,
            "Maximum_Daily_Rainfall.npy"
        ),
        period_max
    )

    np.save(
        os.path.join(
            NPY_DIR,
            "Accumulated_Rainfall_10Days.npy"
        ),
        period_total
    )

    # -------------------------------------------------------------------------
    # Helper plotting function
    # -------------------------------------------------------------------------

    def make_map(
        data,
        title,
        filename,
        label,
        map_vmax
    ):

        (
            data_plot,
            lat_plot,
            lon_plot
        ) = crop_to_valid_geometry(
            data,
            latitude,
            longitude
        )

        data_plot[
            ~np.isfinite(data_plot)
        ] = np.nan

        fig, ax = plt.subplots(
            figsize=(12, 9)
        )

        mesh = ax.pcolormesh(
            lon_plot,
            lat_plot,
            data_plot,
            shading="auto",
            vmin=0,
            vmax=map_vmax
        )

        cbar = plt.colorbar(
            mesh,
            ax=ax,
            pad=0.02,
            shrink=0.85
        )

        cbar.set_label(
            label,
            fontsize=11
        )

        ax.set_xlabel(
            "Longitude (°E)"
        )

        ax.set_ylabel(
            "Latitude (°N)"
        )

        ax.set_title(
            title,
            fontsize=14,
            fontweight="bold"
        )

        ax.grid(
            True,
            alpha=0.3,
            linestyle="--",
            linewidth=0.5
        )

        # Geographic extent
        valid_geo = (
            np.isfinite(lat_plot)
            &
            np.isfinite(lon_plot)
        )

        if np.any(valid_geo):

            ax.set_xlim(
                np.nanmin(
                    lon_plot[
                        valid_geo
                    ]
                ),
                np.nanmax(
                    lon_plot[
                        valid_geo
                    ]
                )
            )

            ax.set_ylim(
                np.nanmin(
                    lat_plot[
                        valid_geo
                    ]
                ),
                np.nanmax(
                    lat_plot[
                        valid_geo
                    ]
                )
            )

        plt.tight_layout()

        plt.savefig(
            filename,
            dpi=DPI,
            bbox_inches="tight"
        )

        plt.close()

        print(
            f"Saved: {filename}"
        )

    # -------------------------------------------------------------------------
    # Mean map
    # -------------------------------------------------------------------------

    make_map(
        period_mean,
        "MOSDAC Mean Daily Precipitation\n"
        "10–19 July 2026",
        os.path.join(
            SUMMARY_DIR,
            "Mean_Daily_Precipitation_10JUL_19JUL_2026.png"
        ),
        "Mean precipitation (mm/day)",
        vmax
    )

    # -------------------------------------------------------------------------
    # Maximum map
    # -------------------------------------------------------------------------

    max_vmax = np.nanpercentile(
        period_max,
        99
    )

    make_map(
        period_max,
        "MOSDAC Maximum Daily Precipitation\n"
        "10–19 July 2026",
        os.path.join(
            SUMMARY_DIR,
            "Maximum_Daily_Precipitation_10JUL_19JUL_2026.png"
        ),
        "Maximum daily precipitation (mm/day)",
        max_vmax
    )

    # -------------------------------------------------------------------------
    # Accumulated map
    # -------------------------------------------------------------------------

    total_vmax = np.nanpercentile(
        period_total,
        99
    )

    make_map(
        period_total,
        "MOSDAC Accumulated Precipitation\n"
        "10–19 July 2026",
        os.path.join(
            SUMMARY_DIR,
            "Accumulated_Precipitation_10JUL_19JUL_2026.png"
        ),
        "Accumulated precipitation (mm)",
        total_vmax
    )


# =============================================================================
# MAIN
# =============================================================================

def main():

    print("\n")
    print("=" * 90)
    print("MOSDAC INSAT HEM DAILY PRECIPITATION ANALYSIS")
    print("=" * 90)

    print(
        f"\nPeriod:"
    )

    print(
        f"{START_DATE} to {END_DATE}"
    )

    # =========================================================================
    # FIND FILES
    # =========================================================================

    files = find_daily_files()

    print("\n")
    print("=" * 90)
    print("UNIQUE HEM FILES FOUND")
    print("=" * 90)

    for i, f in enumerate(
        files,
        start=1
    ):

        print(
            f"{i:02d}. {f}"
        )

    print(
        f"\nTotal unique files: {len(files)}"
    )

    if len(files) == 0:

        raise RuntimeError(
            "No HEM daily files found."
        )

    # =========================================================================
    # COMMON MAP SCALE
    # =========================================================================

    common_vmax = determine_common_vmax(
        files
    )

    print(
        f"\nCommon map scale:"
    )

    print(
        f"0 – {common_vmax:.2f} mm/day"
    )

    # =========================================================================
    # PROCESS FILES
    # =========================================================================

    results = []

    rainfall_arrays = []

    latitude_reference = None

    longitude_reference = None

    processed_dates = []

    for filename in files:

        date = extract_date_from_filename(
            filename
        )

        if date is None:

            print(
                "WARNING: Date could not be extracted."
            )

            continue

        try:

            (
                rainfall,
                latitude,
                longitude
            ) = read_mosdac_file(
                filename
            )

            # -----------------------------------------------------------------
            # Statistics
            # -----------------------------------------------------------------

            stats = calculate_statistics(
                rainfall,
                latitude,
                longitude
            )

            # -----------------------------------------------------------------
            # Save result
            # -----------------------------------------------------------------

            results.append({

                "date":
                    date.strftime(
                        "%Y-%m-%d"
                    ),

                "mean_rainfall_mm_day":
                    stats[
                        "mean_rainfall_mm_day"
                    ],

                "median_rainfall_mm_day":
                    stats[
                        "median_rainfall_mm_day"
                    ],

                "minimum_rainfall_mm_day":
                    stats[
                        "minimum_rainfall_mm_day"
                    ],

                "maximum_rainfall_mm_day":
                    stats[
                        "maximum_rainfall_mm_day"
                    ],

                "std_rainfall_mm_day":
                    stats[
                        "std_rainfall_mm_day"
                    ],

                "valid_pixels":
                    stats[
                        "valid_pixels"
                    ],

                "total_pixels":
                    stats[
                        "total_pixels"
                    ],

                "coverage_percent":
                    stats[
                        "coverage_percent"
                    ],

                "file":
                    filename,

                "rainfall_dataset":
                    RAIN_DATASET,

                "rainfall_units":
                    "mm/day"
            })

            rainfall_arrays.append(
                rainfall
            )

            processed_dates.append(
                date
            )

            if latitude_reference is None:

                latitude_reference = latitude

            if longitude_reference is None:

                longitude_reference = longitude

            # -----------------------------------------------------------------
            # Console statistics
            # -----------------------------------------------------------------

            print("\n")
            print(
                "-" * 70
            )

            print(
                f"DATE: "
                f"{date.strftime('%d %B %Y')}"
            )

            print(
                f"Mean: "
                f"{stats['mean_rainfall_mm_day']:.4f} mm/day"
            )

            print(
                f"Median: "
                f"{stats['median_rainfall_mm_day']:.4f} mm/day"
            )

            print(
                f"Minimum: "
                f"{stats['minimum_rainfall_mm_day']:.4f} mm/day"
            )

            print(
                f"Maximum: "
                f"{stats['maximum_rainfall_mm_day']:.4f} mm/day"
            )

            print(
                f"Valid pixels: "
                f"{stats['valid_pixels']}"
            )

            print(
                f"Coverage: "
                f"{stats['coverage_percent']:.2f}%"
            )

            # -----------------------------------------------------------------
            # Daily map
            # -----------------------------------------------------------------

            map_file = os.path.join(
                DAILY_MAP_DIR,
                f"MOSDAC_Rainfall_"
                f"{date.strftime('%Y%m%d')}.png"
            )

            plot_daily_map(
                rainfall,
                latitude,
                longitude,
                date,
                map_file,
                FIXED_VMIN,
                common_vmax
            )

        except Exception as e:

            print("\n")
            print(
                "ERROR PROCESSING:"
            )

            print(filename)

            print(
                f"Error: {e}"
            )

    # =========================================================================
    # DATAFRAME
    # =========================================================================

    df = pd.DataFrame(
        results
    )

    if df.empty:

        raise RuntimeError(
            "No files were successfully processed."
        )

    df["date"] = pd.to_datetime(
        df["date"]
    )

    df = df.sort_values(
        "date"
    ).reset_index(
        drop=True
    )

    # =========================================================================
    # SAVE CSV
    # =========================================================================

    csv_file = os.path.join(
        CSV_DIR,
        "MOSDAC_HEM_Daily_Rainfall_"
        "10JUL_19JUL_2026.csv"
    )

    df.to_csv(
        csv_file,
        index=False
    )

    # =========================================================================
    # GRAPHS
    # =========================================================================

    plot_mean_timeseries(
        df,
        os.path.join(
            SUMMARY_DIR,
            "Daily_Spatial_Mean_Rainfall.png"
        )
    )

    plot_max_timeseries(
        df,
        os.path.join(
            SUMMARY_DIR,
            "Daily_Spatial_Maximum_Rainfall.png"
        )
    )

    # =========================================================================
    # MULTI-DAY PRODUCTS
    # =========================================================================

    if len(rainfall_arrays) > 0:

        create_multiday_maps(
            rainfall_arrays,
            latitude_reference,
            longitude_reference,
            common_vmax
        )

    # =========================================================================
    # SUMMARY
    # =========================================================================

    summary_file = os.path.join(
        CSV_DIR,
        "Rainfall_Summary.txt"
    )

    with open(
        summary_file,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            "MOSDAC INSAT HEM DAILY PRECIPITATION ANALYSIS\n"
        )

        f.write(
            "=" * 70
            + "\n\n"
        )

        f.write(
            f"Period: {START_DATE} to {END_DATE}\n"
        )

        f.write(
            f"Files processed: {len(df)}\n"
        )

        f.write(
            "Rainfall units: mm/day\n\n"
        )

        f.write(
            f"Mean of daily spatial means: "
            f"{df['mean_rainfall_mm_day'].mean():.4f} mm/day\n"
        )

        f.write(
            f"Maximum daily spatial mean: "
            f"{df['mean_rainfall_mm_day'].max():.4f} mm/day\n"
        )

        highest_mean = df.loc[
            df[
                "mean_rainfall_mm_day"
            ].idxmax()
        ]

        f.write(
            f"Date of maximum spatial mean: "
            f"{highest_mean['date'].strftime('%d %B %Y')}\n"
        )

        f.write(
            f"\nMaximum individual pixel rainfall: "
            f"{df['maximum_rainfall_mm_day'].max():.4f} mm/day\n"
        )

        highest_pixel = df.loc[
            df[
                "maximum_rainfall_mm_day"
            ].idxmax()
        ]

        f.write(
            f"Date of maximum pixel rainfall: "
            f"{highest_pixel['date'].strftime('%d %B %Y')}\n"
        )

        f.write(
            f"\nAccumulated rainfall based on daily "
            f"spatial means: "
            f"{df['mean_rainfall_mm_day'].sum():.4f} mm\n"
        )

    # =========================================================================
    # FINAL OUTPUT
    # =========================================================================

    print("\n")
    print("=" * 90)
    print("PROCESSING COMPLETED SUCCESSFULLY")
    print("=" * 90)

    print("\nDAILY STATISTICS")
    print(
        df[
            [
                "date",
                "mean_rainfall_mm_day",
                "median_rainfall_mm_day",
                "minimum_rainfall_mm_day",
                "maximum_rainfall_mm_day",
                "std_rainfall_mm_day",
                "valid_pixels",
                "coverage_percent"
            ]
        ].to_string(
            index=False
        )
    )

    print("\n")
    print(
        "CSV:"
    )

    print(
        csv_file
    )

    print("\n")
    print(
        "Daily spatial maps:"
    )

    print(
        DAILY_MAP_DIR
    )

    print("\n")
    print(
        "Summary products:"
    )

    print(
        SUMMARY_DIR
    )

    print("\n")
    print(
        "NumPy products:"
    )

    print(
        NPY_DIR
    )

    print("\n")
    print("=" * 90)
    print("DONE")
    print("=" * 90)


# =============================================================================
# RUN
# =============================================================================

if __name__ == "__main__":

    main()



MOSDAC INSAT HEM DAILY PRECIPITATION ANALYSIS

Period:
2026-07-10 to 2026-07-19

Checking folder:
Z:\1\MON FLOOD\MOSDAC_Data\MOSDAC\2026\10JUL
Found 1 HEM file(s).

Checking folder:
Z:\1\MON FLOOD\MOSDAC_Data\MOSDAC\2026\11JUL
Found 1 HEM file(s).

Checking folder:
Z:\1\MON FLOOD\MOSDAC_Data\MOSDAC\2026\12JUL
Found 1 HEM file(s).

Checking folder:
Z:\1\MON FLOOD\MOSDAC_Data\MOSDAC\2026\13JUL
Found 1 HEM file(s).

Checking folder:
Z:\1\MON FLOOD\MOSDAC_Data\MOSDAC\2026\14JUL
Found 1 HEM file(s).

Checking folder:
Z:\1\MON FLOOD\MOSDAC_Data\MOSDAC\2026\15JUL
Found 1 HEM file(s).

Checking folder:
Z:\1\MON FLOOD\MOSDAC_Data\MOSDAC\2026\16JUL
Found 1 HEM file(s).

Checking folder:
Z:\1\MON FLOOD\MOSDAC_Data\MOSDAC\2026\17JUL
Found 1 HEM file(s).

Checking folder:
Z:\1\MON FLOOD\MOSDAC_Data\MOSDAC\2026\18JUL
Found 1 HEM file(s).

Checking folder:
Z:\1\MON FLOOD\MOSDAC_Data\MOSDAC\2026\19JUL
Found 1 HEM file(s).


UNIQUE HEM FILES FOUND
01. Z:\1\MON FLOOD\MOSDAC_Data\MOSDAC\2026\10JUL\3RIM